# 案例：中心控制器编排与领域智能体协作

> Last updated: 2026-11-18

## 核心学习目标

本案例以多城市旅行规划为对象，学习如何设计领域智能体角色，并让中心编排器根据任务能力完成角色匹配、依赖排序和统一调度。

## 案例设计

## 中心编排流程

1. 中心智能体先判断请求类型。非旅行请求直接由大模型回复，不启动后续复杂流程。
2. 旅行请求先复用事实调查框架识别已知条件、待查信息和不确定项，再将目标拆解为子任务并分析依赖。
3. 独立的路由模块根据领域智能体职责、工具能力和输入输出定义，为每个子任务匹配最合适的智能体角色。
4. 中心控制器按依赖顺序调用领域智能体，全部任务完成后由最终规划智能体生成统一答复。

`fact_survey.py` 提供跨章节事实调查模板；`task_plan.py` 负责任务拆解和依赖校验；`agent_routing.py` 独立负责任务到智能体的匹配；`domain_agents.py` 定义领域角色和能力；`orchestrator.py` 用代码控制请求分诊、规划、路由和执行。

**学习重点**：
1. **领域智能体角色设定**：理解如何用职责描述、工具、输入输出和提示词定义一个领域智能体。
2. **任务与智能体角色匹配**：理解拆解后的子任务如何根据能力边界匹配到合适的领域角色。
3. **中心化编排**：理解中心编排器如何完成事实调查、任务拆解、依赖排序、执行调度和最终结果选择。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 模型访问 | 模型创建、文本调用、JSON 与结构化输出 | `llm.py` |
| 请求分诊 | 区分旅行任务与非旅行任务，非旅行任务直接回复 | `orchestrator.py` |
| 事实调查 | 四类事实分析与相对日期换算 | `fact_survey.py` |
| 任务规划 | 任务拆解、依赖分析和执行顺序校验 | `task_plan.py` |
| 角色匹配 | 子任务到领域智能体的能力路由 | `agent_routing.py` |
| 中心编排 | 统一执行、分层调度、最终回复选择与记忆写入 | `orchestrator.py` |
| 领域角色 | 领域智能体的职责、工具、输入输出和提示词 | `domain_agents.py` |
| 长期记忆 | 复用 Chapter-3 的短期上下文、Chroma 检索和记忆写入 | `memory_adapter.py` |
| 最终规划 | 汇总领域结果并选择最终用户答复 | `orchestrator.py` |

## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 Notebook 前，请在仓库根目录执行：

```bash
uv pip install -r Chapter-6/requirements.txt
```

环境变量统一从仓库根目录 `.env` 加载，不在章节目录维护局部 `.env`。至少配置 `DASHSCOPE_API_KEY`；酒店、景点和餐厅工具可选用 `BAIDU_MAP_AK` 或 `AMAP_KEY`。天气和航班工具当前返回空结果。

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

chapter6_candidate = Path.cwd()
if not (chapter6_candidate / "config.py").exists():
    chapter6_candidate = Path.cwd() / "Chapter-6"

if not (chapter6_candidate / "config.py").exists():
    raise FileNotFoundError("Could not locate Chapter-6/config.py")

sys.path.insert(0, str(chapter6_candidate))

from config import BOOK_ROOT, CH6_DIR, CHROMA_DIR, load_project_dotenv

env_path = load_project_dotenv()
print("Chapter-6:", CH6_DIR)
print("项目根目录:", BOOK_ROOT)
print("环境文件:", env_path)
print("Chroma 目录:", CHROMA_DIR)
print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

## 1. 依赖与模块导入

导入中心编排器、任务规划器、领域智能体和共享模块。`llm.py` 负责模型访问，`orchestrator.py` 负责统一控制流程，`domain_agents.py` 提供领域能力。

In [ ]:
import asyncio
import importlib
import json

import fact_survey
import orchestrator
import agent_routing
import map_api
import hotel_search
import restaurant_search
import attraction_search
import memory_adapter
import domain_agents
import task_plan

for mod in (
    map_api,
    hotel_search,
    restaurant_search,
    attraction_search,
    fact_survey,
    agent_routing,
    task_plan,
    memory_adapter,
    domain_agents,
    orchestrator,
):
    importlib.reload(mod)

from orchestrator import CENTRAL_AGENT_SYSTEM_PROMPT, CentralOrchestrator
from domain_agents import DOMAIN_AGENT_SPECS, DomainAgentRegistry, get_domain_agent
from fact_survey import FACTS_PROMPT

print("✓ 模块加载完成（中心编排 / 任务规划 / 领域智能体 / 记忆）")
print(f"领域智能体: {list(DOMAIN_AGENT_SPECS)}")
central_agent_system_prompt = CENTRAL_AGENT_SYSTEM_PROMPT
print(f"中心 System Prompt: {len(central_agent_system_prompt)} 字符")
print(f"FACTS_PROMPT 含 today 占位: {'{today}' in FACTS_PROMPT}")
print(f"memory_adapter 清记忆 API: {hasattr(memory_adapter, 'reset_chroma_directory')}")

## 2. 查看中心智能体的 System Prompt

In [ ]:
from IPython.display import Markdown, display

display(Markdown("### 中心智能体 System Prompt（请求分诊）"))
display(Markdown("该 Prompt 用于区分旅行任务与非旅行任务。非旅行请求直接回复；旅行请求交给由代码控制的规划、路由和执行流程。"))
display(Markdown("```text\n" + central_agent_system_prompt + "\n```"))

display(Markdown("### Chapter-2 预调查 Prompt（节选）"))
display(Markdown("```text\n" + FACTS_PROMPT[:800] + "\n...(截断)...\n```"))

## 3. 查看领域智能体注册表

In [ ]:
from domain_agents import DomainAgentRegistry

registry = DomainAgentRegistry()

print("=== 领域智能体团队 ===\n")
print(registry.get_all_agents_text())

print("\n=== 领域智能体技能参数 ===\n")
print(registry.get_agent_parameters_text())

## 4. 演示：单个领域智能体调用（HotelAgent）

In [ ]:
# 创建酒店推荐领域智能体
from langchain_core.messages import AIMessage, AIMessageChunk, ToolMessage

hotel_agent = get_domain_agent("HotelAgent")

user_query = "我要去大同玩三天，给我推荐酒店，需要安静、近景区"


def _content_to_str(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text") or ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content) if content else ""


def _simplify_tool_output(output):
    if hasattr(output, "content"):
        output = output.content
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return output
    if isinstance(output, dict):
        return {
            "city": output.get("city"),
            "hotels_count": len(output.get("hotels") or []),
            "search_query": output.get("search_query"),
        }
    return output


print(f"用户请求: {user_query}\n")
print("Agent响应: ", end="", flush=True)

seen_tool_starts = set()
ai_text_buffer = ""
config = {"configurable": {"thread_id": "hotel_agent_001"}}

# 新版 create_agent 走 stream_mode=messages，不再稳定触发 on_chat_model_stream
async for msg, _meta in hotel_agent.astream(
    {"messages": [("user", user_query)]},
    config,
    stream_mode="messages",
):
    if isinstance(msg, (AIMessage, AIMessageChunk)):
        for tc in msg.tool_calls or []:
            name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "")
            if name and name not in seen_tool_starts:
                seen_tool_starts.add(name)
                args = tc.get("args") if isinstance(tc, dict) else getattr(tc, "args", {})
                print(f"\n\n>>> 调用工具: {name}")
                print(f">>> 参数: {json.dumps(args, ensure_ascii=False, indent=2)}")
        if msg.tool_calls:
            continue
        text = _content_to_str(msg.content)
        if not text:
            continue
        if text.startswith(ai_text_buffer):
            delta = text[len(ai_text_buffer):]
            ai_text_buffer = text
        else:
            delta = text
            ai_text_buffer += text
        if delta:
            print(delta, end="", flush=True)
    elif isinstance(msg, ToolMessage):
        simplified = _simplify_tool_output(msg.content)
        print(f"\n>>> 工具返回: {json.dumps(simplified, ensure_ascii=False, indent=2)}")
        print("\nAgent继续: ", end="", flush=True)

print()

## 4.5 演示前：清空污染记忆（建议每次重跑 §5 前执行）

**推荐顺序：** §0 → §1 → **本节** → §5 → §6–§8

- 本节会 `reload(memory_adapter)`，并 `close()` 旧 Chroma 客户端（避免 `ValueError: ... different settings`）
- 若 §5 已跑过，会先 `del orchestrator` 再清空
- 若仍失败：**Restart Kernel** → §0 → §1 → 本节

In [ ]:
import gc
import importlib
import sys

# 必须先 reload：kernel 里可能是旧版 memory_adapter（无 release_memory_handles）
import memory_adapter

importlib.reload(memory_adapter)

from config import CHROMA_DIR
from memory_adapter import release_memory_handles, reset_chroma_directory

if not hasattr(memory_adapter, "reset_chroma_directory"):
    raise ImportError(
        "memory_adapter 仍是旧版本。请 Restart Kernel 后：§0 → §1 → 本节。"
    )

ORCHESTRATION_THREAD_ID = "central_orchestration_001"
RESET_CHROMA = True  # 设为 False 可保留历史记忆做对比实验

if RESET_CHROMA:
    if "orchestrator" in globals():
        release_memory_handles(getattr(orchestrator, "memory_adapter", None))
        orchestrator.memory_adapter = None
        del orchestrator
        gc.collect()
    reset_chroma_directory(str(CHROMA_DIR))
    print(f"✓ 已清空 Chroma: {CHROMA_DIR}")
else:
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"ℹ️ 保留现有 Chroma: {CHROMA_DIR}")

print(f"  thread_id 将使用: {ORCHESTRATION_THREAD_ID}（新 orchestrator 实例短期缓冲为空）")
print("→ 下一步请运行 5")

## 5. 演示：中心智能体完整流程

完整旅行链路约 3–8 分钟（请求分诊 + 5 个子任务 + 最终规划）。成功标志：

- 请求分诊结果为旅行任务，非旅行请求不会进入任务规划
- 事实调查结果中的相对日期已换算为明确的 `YYYY-MM-DD`
- Ch3 检索记忆为 **0 条**（§4.5 清空后）或仅与本次相关的条目
- §8 由 ItineraryAgent 汇总依赖结果并生成最终日程

In [ ]:
# Ch2→Ch3→Ch4→Ch5+ 完整流程（请先跑 §1 + §4.5）
import importlib
import fact_survey
import orchestrator
import agent_routing
import map_api
import hotel_search
import restaurant_search
import attraction_search
import domain_agents
import task_plan

for mod in (
    map_api,
    hotel_search,
    restaurant_search,
    attraction_search,
    fact_survey,
    agent_routing,
    task_plan,
    domain_agents,
    orchestrator,
):
    importlib.reload(mod)

from orchestrator import CentralOrchestrator

ORCHESTRATION_THREAD_ID = globals().get(
    "ORCHESTRATION_THREAD_ID", "central_orchestration_001"
)
orchestrator = CentralOrchestrator(enable_memory=True)

test_query = """
你能帮我规划一个下周的多城市旅行吗？我还没想好行程顺序……
大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、
天气情况和美食攻略。我喜欢住安静的酒店，预算每晚不超过800元。
"""

print("=" * 80)
print("🚀 启动中心智能体（分诊→Ch2→Ch3→Ch4→路由→执行）")
print("=" * 80)

result = await orchestrator.process_request(
    test_query, thread_id=ORCHESTRATION_THREAD_ID
)

from IPython.display import Markdown, display
display(Markdown("## ✅ 最终旅行规划\n\n" + result["final_response"]))

## 6. 查看执行计划

In [ ]:
print("=== 思维链预调查 ===\n")
pre_survey = result["execution_plan"]["pre_survey"]
print(json.dumps(pre_survey, ensure_ascii=False, indent=2))

print("\n=== 任务拆解结果 ===\n")
for task in result["execution_plan"]["subtasks"]:
    print(f"[{task['task_id']}] {task['description']}")
    print(f"     智能体: {task['agent']}")
    print(f"     依赖: {task.get('depends_on', [])}")
    print()

print("=== 执行顺序 ===\n")
print(" -> ".join(result["execution_plan"]["execution_order"]))

## 7. 查看子任务执行结果

每项为 `{task_id, agent, tool_data, agent_summary}`：

| 字段 | 含义 |
|------|------|
| `tool_data` | 工具原始 JSON（酒店列表、天气、行程 `plan` 等） |
| `agent_summary` | 领域 Agent 在**工具返回之后**的文字总结 |

**若出现「我只能协助…」且 `tool_data` 为空**：领域 Agent 把中心编排的长描述误判为「非本域问题」而拒答（例如 T2 描述写「三地酒店」但 params 只有 `city: 上海`）。编排器已改为强调 **结构化参数优先** 下发；修改代码后请 **重新运行 §1 导入 + §5 完整流程**。

**若 `tool_data` 有 `plan` 等字段但 summary 仍很短**：以前可能取错了最后一条 AI 消息；现已优先取工具调用后的总结。

In [ ]:
def _brief_tool_data(data, max_items=3):
    """从 subtask_results[*].tool_data 提取可读摘要。"""
    if data is None:
        return ["  工具数据: （未调用工具或无返回）"]
    if isinstance(data, str):
        try:
            data = json.loads(data)
        except json.JSONDecodeError:
            return [f"  工具数据: {data[:200]}{'…' if len(data) > 200 else ''}"]
    if not isinstance(data, dict):
        return [f"  工具数据: {str(data)[:200]}"]

    if data.get("error"):
        return [f"  工具错误: {data['error']}"]

    lines = []
    if hotels := data.get("hotels"):
        lines.append(
            f"  酒店: {len(hotels)} 家 | 搜索: {data.get('search_query') or data.get('city')} | 来源: {data.get('data_source', '?')}"
        )
        for h in hotels[:max_items]:
            if isinstance(h, dict):
                price = h.get("avg_price_cny") or h.get("avg_price")
                rating = h.get("rating")
                extra = f" ¥{price}/晚" if price else ""
                extra += f" 评分{rating}" if rating else ""
                lines.append(f"    · {h.get('name')}{extra}")
    elif restaurants := data.get("restaurants"):
        lines.append(f"  餐厅: {len(restaurants)} 家 | 来源: {data.get('data_source', '?')}")
        for r in restaurants[:max_items]:
            if isinstance(r, dict):
                lines.append(f"    · {r.get('name')} ({r.get('district') or r.get('address', '')})")
    elif attractions := data.get("attractions"):
        lines.append(f"  景点: {len(attractions)} 个 | 来源: {data.get('data_source', '?')}")
        for a in attractions[:max_items]:
            if isinstance(a, dict):
                lines.append(f"    · {a.get('name')}")
    elif flights := data.get("flights"):
        lines.append(f"  航班: {len(flights)} 条 | {data.get('departure')}→{data.get('arrival')} {data.get('date')}")
        for f in flights[:max_items]:
            if isinstance(f, dict):
                lines.append(f"    · {f.get('flight_no') or f.get('number')} {f.get('departure_time')}–{f.get('arrival_time')}")
    elif forecast := data.get("forecast"):
        city = data.get("city", "")
        date = data.get("date", "")
        if isinstance(forecast, dict):
            cond = forecast.get("condition") or forecast.get("dayweather") or forecast.get("text")
            hi = forecast.get("high") or forecast.get("daytemp")
            lo = forecast.get("low") or forecast.get("nighttemp")
            lines.append(f"  天气: {city} {date} | {cond} | 气温 {lo}~{hi}°C")
        else:
            lines.append(f"  天气: {city} {date} | {str(forecast)[:120]}")
    elif data.get("text"):
        lines.append(f"  天气摘要: {data.get('text')}")
    elif data.get("plan") or data.get("daily_plan"):
        plan = data.get("plan") or data.get("daily_plan")
        days = data.get("days") or (len(plan) if isinstance(plan, list) else "?")
        lines.append(
            f"  行程: {days} 天 | {data.get('departure_city')}→{data.get('destination_city')}"
        )
        if isinstance(plan, list) and plan:
            day1 = plan[0]
            if isinstance(day1, dict):
                lines.append(f"    · Day1: {day1.get('theme') or day1.get('title') or str(day1)[:80]}")
    elif data.get("itinerary"):
        plan = data.get("itinerary")
        n_days = len(plan) if isinstance(plan, list) else 1
        lines.append(f"  行程: {n_days} 天规划 | {data.get('departure_city')}→{data.get('destination_city')}")
    else:
        lines.append(f"  工具字段: {list(data.keys())}")
    return lines


def _brief_summary(text, limit=280):
    if not text:
        return "  Agent 总结: （无文字回复，请看工具数据）"
    one_line = " ".join(str(text).split())
    if len(one_line) > limit:
        one_line = one_line[:limit] + "…"
    return f"  Agent 总结: {one_line}"


print("=== 子任务执行结果 ===\n")
print("结构说明: 每项含 agent / tool_data（工具原始结果）/ agent_summary（领域 Agent 文字总结）\n")

for task_id, task_result in result["subtask_results"].items():
    print(f"[{task_id}] {task_result.get('agent', '?') if isinstance(task_result, dict) else task_id}")
    if isinstance(task_result, dict):
        for line in _brief_tool_data(task_result.get("tool_data")):
            print(line)
        print(_brief_summary(task_result.get("agent_summary")))
    else:
        print(f"  结果: {task_result}")
    print()

## 8. 查看最终回复

In [ ]:
print("=" * 80)
print("✅ 中心智能体最终回复")
print("=" * 80)
print()
print(result["final_response"])

## 9. 对比实验：非旅行直答、简单旅行查询 vs 复杂查询

In [ ]:
# 非旅行请求：中心智能体直接回复，不进入任务规划与智能体调度
direct_query = "请用一句话解释什么是二分查找。"

direct_result = await orchestrator.process_request(
    direct_query, thread_id="direct_query_001"
)

assert direct_result["route"] == "direct"
assert direct_result["execution_plan"] is None
print("直接回复:")
print(direct_result["final_response"])

In [ ]:
# 简单查询：只需调用一个领域智能体
simple_query = "查询上海明天的天气"

print("=" * 80)
print("测试用例：简单查询")
print("=" * 80)
print(f"用户请求: {simple_query}\n")

simple_result = await orchestrator.process_request(
    simple_query, thread_id="simple_query_001"
)

print("\n执行计划:")
print(json.dumps(simple_result["execution_plan"]["subtasks"], ensure_ascii=False, indent=2))

print("\n最终回复:")
print(simple_result["final_response"])

## 10. 小结

**本章学到的核心概念**：

1. **中心智能体请求分诊**
   - 使用中心 System Prompt 区分旅行任务与非旅行任务
   - 非旅行任务直接回复，不启动复杂多智能体流程
   - 旅行任务进入由代码控制的规划、路由和执行流程

2. **任务规划**
   - `task_plan.py` 完成事实调查、任务拆解和依赖分析
   - 任务规划不负责选择领域智能体
   - 输出包含任务、依赖关系和执行顺序的结构化计划

3. **领域智能体路由与执行**
   - `agent_routing.py` 独立完成任务到领域智能体的匹配
   - 中心控制器按依赖顺序分层调度领域智能体
   - 支持无依赖任务的并行执行

4. **最终规划与总结**
   - 领域智能体返回天气、景点、酒店、餐厅和航班等事实结果
   - ItineraryAgent 综合依赖结果生成最终日程与用户答复
   - 工具未返回的信息不得补造

**实际应用建议**：
- 可以根据业务需求添加更多领域智能体（如交通、签证、保险等）
- 集成真实的长期记忆系统，实现个性化推荐
- 优化任务拆解策略，提高执行效率
- 添加错误处理和重试机制，增强系统鲁棒性